# YOLO PCX Notebook (organized)

This notebook is structured into clear sections: setup → dataset/model → clustering → visualization.
Optional/debug cells are kept at the end.

## 1) Project paths and environment

In [1]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path)

['/home/jovyan/FHHI-XAI', '/opt/conda/lib/python311.zip', '/opt/conda/lib/python3.11', '/opt/conda/lib/python3.11/lib-dynload', '', '/opt/conda/lib/python3.11/site-packages']


In [2]:
!pip install joblib
!pip install zennit-crp
!pip install segmentation_models_pytorch
!pip install addict
!pip install natsort
!pip install umap-learn

## 2) Imports

In [3]:
# === Standard Library ===
import os
import sys
import copy
import h5py
from PIL import Image 
import os, matplotlib.pyplot as plt 
import cv2
import torch
import numpy as np
from yolov6.data.data_augment import letterbox
import math
from PIL import Image 
import os, matplotlib.pyplot as plt 
import cv2
import torch
import numpy as np
from yolov6.data.data_augment import letterbox
import math

# === Scientific Computing ===
import numpy as np
from sklearn.mixture import GaussianMixture

# === Torch & TorchVision ===
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from torchvision.utils import (
    draw_segmentation_masks,
    draw_bounding_boxes,
    make_grid
)

# === PIL & Plotting ===
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

# === Progress Bar ===
from tqdm import tqdm

# === CRP & Zennit ===
import zennit.image as zimage
from crp.image import imgify
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names
from PIL import Image 
import os, matplotlib.pyplot as plt 
import torch
import numpy as np
from yolov6.data.data_augment import letterbox
import math
# === LCRP Utilities ===
from LCRP.models import get_model
from LCRP.utils.crp_configs import ATTRIBUTORS, CANONIZERS, VISUALIZATIONS, COMPOSITES

# === Local Project Modules ===
sys.path.append("..")  # Temporary path extension for imports

#from src.minio_client import MinIOClient
from src.glocal_analysis import run_analysis 
from src.datasets.person_car_dataset import PersonCarDataset
from src.yolo_pcx_test_fire import plot_pcx_explanations
from src.letterbox_utils import letterbox_transform, check_img_size, rescale_boxes

# --- FIRST CELL (before importing umap/numba!) ---
import os, logging

# Ensure env isn’t forcing DEBUG
os.environ["NUMBA_LOG_LEVEL"] = "WARNING"   # or "ERROR"/"CRITICAL"
os.environ.pop("NUMBA_DEBUG", None)

# Hard‐mute numba loggers and stop propagation to root
for name in ("numba", "numba.core", "numba.core.ssa"):
    lg = logging.getLogger(name)
    lg.setLevel(logging.ERROR)      # try WARNING/ERROR/CRITICAL
    lg.propagate = False
    # remove any existing noisy handlers
    for h in list(lg.handlers):
        lg.removeHandler(h)
    lg.addHandler(logging.NullHandler())

# --- Silence Matplotlib DEBUG logging ---
import os, logging
import matplotlib as mpl

# Just in case someone exported this
os.environ.pop("MPLDEBUG", None)

# Matplotlib's own switch
try:
    mpl.set_loglevel("warning")   # or "error"
except Exception:
    pass

# Force all matplotlib loggers to WARNING and stop propagation to root
for name in ("matplotlib", "matplotlib.font_manager"):
    lg = logging.getLogger(name)
    lg.setLevel(logging.WARNING)   # or logging.ERROR
    lg.propagate = False
    # Remove any existing noisy handlers (e.g., StreamHandler at DEBUG)
    for h in list(lg.handlers):
        lg.removeHandler(h)
    # Add a NullHandler so nothing leaks upward
    lg.addHandler(logging.NullHandler())

# --- Silence PIL/Pillow DEBUG logs ---
import os, logging
os.environ.pop("PILLOW_DEBUG", None)  # just in case

# Force every PIL logger to WARNING (or ERROR) and stop propagation
for name in [n for n in logging.root.manager.loggerDict if n == "PIL" or n.startswith("PIL.")]:
    lg = logging.getLogger(name)
    lg.setLevel(logging.WARNING)      # or logging.ERROR
    lg.propagate = False
    for h in list(lg.handlers):       # remove any noisy handlers
        lg.removeHandler(h)
    lg.addHandler(logging.NullHandler())


## 3) Dataset loading

In [4]:
dtype = torch.float32
root_dir = "/home/jovyan/FHHI-XAI/data/RAS_TrialData"

from functools import partial
dataset = PersonCarDataset(root_dir=root_dir, split="train", transform=None)

## 4) Model loading

In [5]:
model_name = "yolov6s6"
ckpt_path = "/home/jovyan/FHHI-XAI/models/best_ckpt.pt"

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cuda:5"
print("Using device:", device)

# Loading unet with path to checkpoint
model = get_model(model_name=model_name, classes=2, ckpt_path=ckpt_path, device=device, dtype=dtype)
model = model.to(device)

Using device: cuda:5


### Quick sanity checks (optional)

In [6]:
print(len(dataset))

558


## 5 Extract PCX attributions per detection and save artifacts


In [7]:
# # %% PCX per-detection extraction & save (class-gated) — TILE-BASED (same as CRP)
# # Processes ALL Conv2d layers in the model
# import os, json, numpy as np, torch
# from pathlib import Path
# from functools import partial
# from PIL import Image
import torchvision.transforms.functional as TF
# from crp.concepts import ChannelConcept
# from crp.helper import get_layer_names
# from LCRP.utils.crp_configs import ATTRIBUTORS, CANONIZERS, COMPOSITES

# # -------- config --------
# THRESH        = 0.4
# CLASSES       = [0, 1]

# OUT_BASE      = f"../output_RAS/pcx/yolo_person_car"

# MAX_TILES_PER_CLASS = {0: 100000, 1: 100000}

# -------- helpers --------
def _to_probs_safe(scores: torch.Tensor) -> torch.Tensor:
    if scores.numel() == 0:
        return scores
    if torch.all((scores >= 0) & (scores <= 1)):
        return scores
    return torch.softmax(scores, dim=-1)

def _detach_like(x):
    if torch.is_tensor(x):
        return x.detach()
    if isinstance(x, (list, tuple)):
        return type(x)(_detach_like(t) for t in x)
    return x

# -------- tile dataset (same as CRP) --------
class ExhaustiveTileDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset, tile_size=640, stride=576):
        self.base_dataset = base_dataset
        self.tile_size = tile_size
        self.stride = stride
        self.class_names = getattr(base_dataset, 'class_names', ['person', 'car'])
        self.tile_mapping = []
        print("Scanning dataset to build exhaustive tile map...")
        for img_idx in range(len(base_dataset)):
            out = base_dataset[img_idx]
            img = out[0] if isinstance(out, (list, tuple)) else out
            if isinstance(img, torch.Tensor): H, W = img.shape[-2:]
            elif isinstance(img, Image.Image): W, H = img.size
            else: W, H = Image.fromarray(np.array(img)).size
            xs = sorted(set(list(range(0, W, stride)) + [max(0, W - tile_size)]))
            ys = sorted(set(list(range(0, H, stride)) + [max(0, H - tile_size)]))
            for y0 in ys:
                for x0 in xs:
                    self.tile_mapping.append((img_idx, x0, y0))
        print(f"Done! {len(self.tile_mapping)} tiles from {len(base_dataset)} images.")
    def __len__(self): return len(self.tile_mapping)
    def __getitem__(self, idx):
        img_idx, x0, y0 = self.tile_mapping[idx]
        out = self.base_dataset[img_idx]
        img = out[0] if isinstance(out, (list, tuple)) else out
        target = out[1] if isinstance(out, (list, tuple)) else None
        if isinstance(img, torch.Tensor): img_pil = TF.to_pil_image(img)
        elif isinstance(img, Image.Image): img_pil = img
        else: img_pil = Image.fromarray(np.array(img))
        tile_pil = img_pil.crop((x0, y0, x0 + self.tile_size, y0 + self.tile_size))
        return TF.to_tensor(tile_pil), target
    def reverse_normalization(self, x):
        if hasattr(self.base_dataset, 'reverse_normalization'): return self.base_dataset.reverse_normalization(x)
        return x
    def __getattr__(self, name): return getattr(self.base_dataset, name)

exhaustive_dataset = ExhaustiveTileDatasetWrapper(dataset)

# # -------- setup model --------
# device = next(model.parameters()).device
# model.eval().to(device)

# attribution = ATTRIBUTORS[model_name](model)
# composite   = COMPOSITES[model_name](canonizers=[CANONIZERS[model_name]()])
# cc = ChannelConcept()

# # GET ALL Conv2d LAYERS
# TARGET_LAYERS = get_layer_names(model, [torch.nn.Conv2d])
# TARGET_LAYERS.remove("module.detect.proj_conv")
# print(f"Found {len(TARGET_LAYERS)} Conv2d layers to process:")
# for i, ln in enumerate(TARGET_LAYERS):
#     print(f"  {i:3d}: {ln}")

# # Storage: vecs[cls][layer] = list of vectors, meta[cls][layer] = list of detection info
# vecs = {cls: {ln: [] for ln in TARGET_LAYERS} for cls in CLASSES}
# meta = {cls: {ln: [] for ln in TARGET_LAYERS} for cls in CLASSES}

# tiles_used = {c: 0 for c in CLASSES}
# kept_dets  = {c: 0 for c in CLASSES}

# def _limits_reached():
#     return all(tiles_used[c] >= MAX_TILES_PER_CLASS.get(c, 10) for c in CLASSES)

# # -------- main loop (iterate over tiles) --------
# print(f"\nStarting extraction for {len(TARGET_LAYERS)} layers over {len(exhaustive_dataset)} tiles...")
# for tile_idx in range(len(exhaustive_dataset)):
#     if _limits_reached():
#         print("✅ Per-class tile limits reached. Stopping.")
#         break

#     if tile_idx % 200 == 0:
#         print(f"Processing tile {tile_idx}/{len(exhaustive_dataset)}...")

#     # Get tile (already 640x640, no letterbox needed)
#     tile_tensor, _ = exhaustive_dataset[tile_idx]
#     img_idx, x0, y0 = exhaustive_dataset.tile_mapping[tile_idx]

#     # --- detection forward ---
#     x_det = tile_tensor.unsqueeze(0).to(device)
#     with torch.enable_grad():
#         x_det.requires_grad_(True)
#         scores_all, boxes_all = model.predict_with_boxes(x_det)

#     scores_all, boxes_all = _detach_like(scores_all), _detach_like(boxes_all)

#     if scores_all is None or boxes_all is None:
#         continue
#     scores = scores_all[0] if scores_all.ndim == 3 else scores_all
#     boxes  = boxes_all[0]  if boxes_all.ndim  == 3 else boxes_all
#     if scores.numel() == 0 or boxes.numel() == 0:
#         continue

#     probs     = _to_probs_safe(scores)
#     label_top = probs.argmax(dim=1)

#     # For each class
#     for cls in CLASSES:
#         if tiles_used[cls] >= MAX_TILES_PER_CLASS.get(cls, 10):
#             continue

#         kept_idx = ((label_top == cls) & (probs[:, cls] >= THRESH)).nonzero(as_tuple=False).flatten().tolist()
#         if not kept_idx:
#             continue

#         tiles_used[cls] += 1

#         # --- attribution per kept detection ---
#         x_attr = tile_tensor.unsqueeze(0).to(device).requires_grad_(True)
#         for box_idx in kept_idx:
#             attribution.take_prediction = int(box_idx)

#             out = attribution(
#                 x_attr,
#                 [{"y": int(cls)}],
#                 composite,
#                 record_layer=TARGET_LAYERS,
#                 init_rel=1
#             )

#             # Box is already in tile coordinates (640x640), no rescaling needed
#             box_tile = boxes[box_idx].detach().cpu().tolist()

#             detection_meta = {
#                 "dataset_idx":  int(tile_idx),
#                 "img_idx":      int(img_idx),
#                 "tile_x0":      int(x0),
#                 "tile_y0":      int(y0),
#                 "box_idx":      int(box_idx),
#                 "cls":          int(cls),
#                 "conf":         float(probs[box_idx, cls].item()),
#                 "box":          box_tile,
#             }

#             # Extract channel concept vectors for EACH target layer
#             for ln in TARGET_LAYERS:
#                 v = cc.attribute(out.relevances[ln], abs_norm=True)[0].detach().cpu().numpy()
#                 vecs[cls][ln].append(v)
#                 meta[cls][ln].append(detection_meta.copy())

#             kept_dets[cls] += 1

#         # Clean up hooks
#         for meth in ("remove_hooks", "_remove_hooks", "clear_hooks"):
#             if hasattr(attribution, meth):
#                 try:
#                     getattr(attribution, meth)()
#                 except Exception:
#                     pass
#                 break

# # -------- summary --------
# print("\n---- Summary ----")
# for c in CLASSES:
#     print(f"class {c}: tiles used = {tiles_used[c]} | detections saved = {kept_dets[c]} | limit = {MAX_TILES_PER_CLASS[c]}")

# print(f"\nTotal layers processed: {len(TARGET_LAYERS)}")

# # -------- VALIDATION: Check alignment --------
# print("\n---- Validation (sample) ----")
# for cls in CLASSES:
#     for ln in [TARGET_LAYERS[0], TARGET_LAYERS[-1]]:
#         n_vecs = len(vecs[cls][ln])
#         n_meta = len(meta[cls][ln])
#         status = "✓" if n_vecs == n_meta else f"✗ MISMATCH!"
#         print(f"class {cls} | {ln}: {n_vecs} vectors, {n_meta} meta entries {status}")

# # -------- save (per-class, per-layer) --------
# print("\n---- Saving ----")
# OUT_BASE = Path(OUT_BASE)
# for cls in CLASSES:
#     for ln in TARGET_LAYERS:
#         out_dir = OUT_BASE / ln
#         out_dir.mkdir(parents=True, exist_ok=True)

#         arr = np.asarray(vecs[cls][ln], dtype=np.float32)
#         np.save(out_dir / f"attributions_{cls}.npy", arr)

#         meta_path = out_dir / f"meta_class_{cls}.json"
#         with open(meta_path, "w") as f:
#             json.dump(meta[cls][ln], f, indent=2)

# print(f"\n✅ Done. Saved data for {len(TARGET_LAYERS)} layers × {len(CLASSES)} classes")
# print(f"   Output directory: {OUT_BASE}")

Scanning dataset to build exhaustive tile map...
Done! 18090 tiles from 558 images.


## 6) PCX explanations visualization

### 6A) Load saved image with path


In [8]:
# import os
# from PIL import Image
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from crp.helper import get_layer_names

# # ============ CONFIGURATION ============
# n_prototypes_by_layer = {
#     "module.backbone.stem.rbr_dense.conv":   {0: 3, 1: 4},
#     "module.backbone.stem.rbr_1x1.conv":     {0: 3, 1: 2},
#     "module.backbone.ERBlock_2.0.rbr_dense.conv":        {0: 3, 1: 9},
#     "module.backbone.ERBlock_2.1.conv1.rbr_dense.conv":  {0: 3, 1: 9},
#     "module.backbone.ERBlock_3.0.rbr_dense.conv":        {0: 3, 1: 9},
#     "module.backbone.ERBlock_3.1.block.0.rbr_1x1.conv":  {0: 3, 1: 9},
#     "module.backbone.ERBlock_6.2.cspsppf.cv7.block.conv": {0: 3, 1: 9},
#     "module.neck.reduce_layer0.block.conv":              {0: 3, 1: 9},
# }

# layer_names = list(n_prototypes_by_layer.keys())

# use_half = False

# # ============ SPECIFY IMAGE PATH HERE ============
# image_path = "/home/said/dev_v1/FHHI-XAI/YOLOV6/data/synthetic/images/train/45844cab-DJI_20211023131615_0007_Z_A.JPG" 

# # ============ LOAD IMAGE ============
# print(f"Loading image from: {image_path}")

# if not os.path.exists(image_path):
#     raise FileNotFoundError(f"Image not found: {image_path}")

# # Load image as PIL
# orig_img = Image.open(image_path).convert("RGB")
# orig_np = np.array(orig_img)
# original_shape = orig_np.shape[:2]  # (H, W)

# print(f"Original image shape (HxW): {original_shape}")

# # ============ MODEL SETUP ============
# model = model.to(device)
# model.eval()
# stride = int(model.stride.max()) if hasattr(model, 'stride') else 64
# img_size = check_img_size(640, stride=stride)

# print(f"Using image size: {img_size}, stride: {stride}")
# print(f"\n✓ Processing {len(layer_names)} layers:")
# for ln in layer_names:
#     print(f"  - {ln}: {n_prototypes_by_layer[ln]}")

# print(f"\n{'#'*70}")
# print(f"# IMAGE: {os.path.basename(image_path)}")
# print(f"{'#'*70}")

# # ============ PREPROCESS IMAGE ============
# img_letterbox = letterbox(orig_np, new_shape=img_size, stride=stride, auto=True)[0]
# letterbox_shape = img_letterbox.shape[:2]

# print(f"Letterbox image shape (HxW): {letterbox_shape}")

# # Convert to tensor
# img_letterbox_transposed = img_letterbox.transpose((2, 0, 1))
# img_tensor = torch.from_numpy(np.ascontiguousarray(img_letterbox_transposed))
# img_tensor = img_tensor.float() if not use_half else img_tensor.half()
# img_tensor /= 255.0
# img_tensor = img_tensor.to(device)

# print(f"Preprocessed tensor shape: {img_tensor.shape}")

# # ============ RUN PREDICTION ============
# img_with_batch = img_tensor.unsqueeze(0)
# with torch.no_grad():
#     scores, boxes = model.predict_with_boxes(img_with_batch)

# # Rescale boxes to original coordinates
# boxes_original = rescale_boxes(
#     boxes[0].cpu().detach().numpy(),
#     letterbox_shape=letterbox_shape,
#     original_shape=original_shape
# )

# num_boxes = boxes_original.shape[0]
# class_ids = scores[0].argmax(dim=1)
# confidences = scores[0].max(dim=1).values

# print(f"\n✓ Found {num_boxes} detections:")

# # Print all detections
# for i in range(num_boxes):
#     cls = class_ids[i].item()
#     conf = confidences[i].item()
#     box = boxes_original[i]
#     print(f"  Detection {i}: class={cls}, conf={conf:.3f}, box=[{box[0]:.1f}, {box[1]:.1f}, {box[2]:.1f}, {box[3]:.1f}]")

# if num_boxes == 0:
#     print("⚠️ No detections, stopping")
# else:
#     # ============ LOOP THROUGH ALL DETECTIONS ============
#     for prediction_num in range(num_boxes):
#         class_id = class_ids[prediction_num].item()
#         conf = confidences[prediction_num].item()
#         box_original = boxes_original[prediction_num]

#         print(f"\n{'#'*70}")
#         print(f"# DETECTION {prediction_num}/{num_boxes-1}: class={class_id}, conf={conf:.3f}")
#         print(f"{'#'*70}")

#         # ============ LOOP THROUGH LAYERS ============
#         for layer_idx, layer_name in enumerate(layer_names):
#             print(f"\n{'='*60}")
#             print(f"LAYER {layer_idx}/{len(layer_names)-1}: {layer_name}")
#             print(f"{'='*60}")

#             prototype_dict = n_prototypes_by_layer[layer_name]
#             print(f"  Prototypes: {prototype_dict}")

#             if class_id not in prototype_dict:
#                 print(f"  ⚠️ class={class_id} not in prototype_dict, skipping")
#                 continue

#             print(f"  n_prototypes for class {class_id}: {prototype_dict[class_id]}")

#             safe_layer = layer_name.replace(".", "_")
#             out_dir = os.path.join(f"../output_{dataset_type}/pcx/pcx_plots", safe_layer)
#             os.makedirs(out_dir, exist_ok=True)

#             try:
#                 # Pass CPU tensor since plot_pcx_explanations may use CPU
#                 img_tensor_cpu = img_tensor.cpu()

#                 fig = plot_pcx_explanations(
#                     model_name=model_name,
#                     model=model,
#                     img=img_tensor_cpu,  # CPU tensor
#                     orig_img=orig_img,   # PIL Image
#                     dataset=dataset,
#                     orig_dataset=orig_dataset,
#                     class_id=class_id,
#                     n_concepts=3,
#                     n_refimgs=12,
#                     num_prototypes=prototype_dict,
#                     prediction_num=prediction_num,
#                     layer_name=layer_name,
#                     ref_imgs_path=f"../output_{dataset_type}/ref_imgs/",
#                     output_dir_pcx=f"../output_{dataset_type}/pcx/yolo_person_car",
#                     output_dir_crp=output_dir_crp,
#                     letterbox_shape=letterbox_shape,
#                     original_shape=original_shape,
#                     rescale_boxes_fn=rescale_boxes,
#                     dataset_type=dataset_type
#                 )

#                 # Move model back to CUDA after plot_pcx_explanations
#                 model = model.to(device)

#                 if fig is not None:
#                     img_name = os.path.splitext(os.path.basename(image_path))[0]
#                     fname = f"{img_name}_det{prediction_num}_class{class_id}_layer{safe_layer}.png"
#                     out_path = os.path.join(out_dir, fname)
#                     fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor='white')
#                     plt.close(fig)
#                     print(f"    ✅ Saved: {out_path}")
#                 else:
#                     print(f"    ⚠️ No figure returned")

#             except Exception as e:
#                 print(f"    ❌ Error: {e}")
#                 import traceback
#                 traceback.print_exc()
#                 # Ensure model is back on CUDA even after error
#                 model = model.to(device)
#                 continue

# print(f"\n{'#'*70}")
# print(f"# ✓ DONE - Processed {num_boxes} detections across {len(layer_names)} layers")
# print(f"# Total explanations generated: {num_boxes * len(layer_names)}")
# print(f"{'#'*70}")

### 6B) Run inference on a selected image from the dataset and visualize detections


In [12]:
import os
from PIL import Image
import numpy as np
import torch
import matplotlib.pyplot as plt
from crp.helper import get_layer_names

# ============ CONFIGURATION ============
n_prototypes_by_layer = {
    # "module.backbone.stem.rbr_dense.conv":   {0: 4, 1: 4},
    "module.backbone.ERBlock_2.0.rbr_dense.conv":        {0: 4, 1: 4},
    "module.backbone.ERBlock_3.0.rbr_dense.conv":        {0: 4, 1: 4},
    # "module.backbone.ERBlock_5.0.rbr_dense.conv": {0: 4, 1: 4}
}

layer_names = list(n_prototypes_by_layer.keys())
use_half = False

# ============ SPECIFY TILE INDEX HERE ============
# tile_idx maps to exhaustive_dataset[tile_idx]
tile_idx = 600
prediction_num = 0

# ============ ENSURE MODEL IS ON CORRECT DEVICE ============
model = model.to(device)
model.eval()

print(f"Using device: {device}")
print(f"\n✓ Processing {len(layer_names)} layers:")
for ln in layer_names:
    print(f"  - {ln}: {n_prototypes_by_layer[ln]}")

print(f"\n{'#'*70}")
print(f"# TILE {tile_idx}")
print(f"{'#'*70}")

# ============ GET TILE (same as CRP) ============
tile_tensor, label = exhaustive_dataset[tile_idx]
img_idx, x0, y0 = exhaustive_dataset.tile_mapping[tile_idx]

print(f"Tile from image {img_idx}, position ({x0}, {y0})")
print(f"Tile tensor shape: {tile_tensor.shape}")

# Get original full image for context
orig_out = dataset[img_idx]
orig_img_raw = orig_out[0] if isinstance(orig_out, (list, tuple)) else orig_out
if torch.is_tensor(orig_img_raw):
    orig_img = Image.fromarray((orig_img_raw.permute(1, 2, 0).numpy() * 255).astype(np.uint8)) if orig_img_raw.dtype != torch.uint8 else Image.fromarray(orig_img_raw.permute(1, 2, 0).numpy())
elif isinstance(orig_img_raw, np.ndarray):
    orig_img = Image.fromarray(orig_img_raw)
else:
    orig_img = orig_img_raw

original_shape = (orig_img.size[1], orig_img.size[0])  # (H, W)
print(f"Original image shape (HxW): {original_shape}")

# Run prediction on tile (already 640x640, no letterbox needed)
img_with_batch = tile_tensor.unsqueeze(0).to(device).requires_grad_(True)
with torch.enable_grad():
    scores, boxes = model.predict_with_boxes(img_with_batch)
scores, boxes = scores.detach(), boxes.detach()


# Boxes are already in tile coordinates (640x640), no rescaling needed
boxes_np = boxes[0].cpu().detach().numpy()
num_boxes = boxes_np.shape[0]
class_ids = scores[0].argmax(dim=1)
confidences = scores[0].max(dim=1).values

print(f"\n✓ Found {num_boxes} detections")

if num_boxes == 0:
    print("⚠️ No detections, stopping")
else:
    # ============ LOOP THROUGH LAYERS ============
    for layer_idx, layer_name in enumerate(layer_names):
        print(f"\n{'='*60}")
        print(f"LAYER {layer_idx}/{len(layer_names)-1}: {layer_name}")
        print(f"{'='*60}")

        prototype_dict = n_prototypes_by_layer[layer_name]
        print(f"  Prototypes: {prototype_dict}")

        safe_layer = layer_name.replace(".", "_")
        out_dir = os.path.join(f"../output_RAS/pcx/pcx_plots", safe_layer)
        os.makedirs(out_dir, exist_ok=True)

        # ============ LOOP THROUGH DETECTIONS ============
        class_id = class_ids[prediction_num].item()
        conf = confidences[prediction_num].item()

        # ============ LOOP THROUGH DETECTIONS ============
        for p_num in range(num_boxes):
            class_id = class_ids[p_num].item()
            conf = confidences[p_num].item()

            if class_id not in prototype_dict:
                print(f"\n  Detection {p_num}: class={class_id} not in prototype_dict, skipping")
                continue

            print(f"\n  Detection {p_num}/{num_boxes-1}: "
                  f"class={class_id}, conf={conf:.3f}, n_prototypes={prototype_dict[class_id]}")

            try:
                img_tensor_cpu = tile_tensor.cpu()

                fig = plot_pcx_explanations(
                    model_name=model_name,
                    model=model,
                    img=img_tensor_cpu,
                    orig_img=orig_img,
                    dataset=exhaustive_dataset,
                    orig_dataset=dataset,
                    class_id=class_id,
                    n_concepts=3,
                    n_refimgs=12,
                    num_prototypes=prototype_dict,
                    prediction_num=p_num, 
                    layer_name=layer_name,
                    ref_imgs_path="../output_RAS/ref_imgs/",
                    output_dir_pcx="../output_RAS/pcx/yolo_person_car",
                    output_dir_crp="../output_RAS/crp/yolo_person_car",
                    letterbox_shape=(640, 640),       # tile is already 640x640
                    original_shape=original_shape,
                    rescale_boxes_fn=lambda b, **kw: b  # no-op: boxes already in tile coords            
                )

                model = model.to(device)
    
                if fig is not None:
                    fname = f"tile{tile_idx:04d}_box{p_num}_class{class_id}_layer{safe_layer}.png"
                    out_path = os.path.join(out_dir, fname)
                    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor='white')
                    plt.close(fig)
                    print(f"    ✅ Saved: {out_path}")
                else:
                    print(f"    ⚠️ No figure returned")
    
            except Exception as e:
                print(f"    ❌ Error: {e}")
                import traceback
                traceback.print_exc()
                model = model.to(device)
                continue

print(f"\n{'#'*70}")
print(f"# ✓ DONE processing tile_idx={tile_idx}")
print(f"{'#'*70}")

Using device: cuda:5

✓ Processing 2 layers:
  - module.backbone.ERBlock_2.0.rbr_dense.conv: {0: 4, 1: 4}
  - module.backbone.ERBlock_3.0.rbr_dense.conv: {0: 4, 1: 4}

######################################################################
# TILE 600
######################################################################
Tile from image 9, position (3416, 1728)
Tile tensor shape: torch.Size([3, 640, 640])
Original image shape (HxW): (3040, 4056)
torch.Size([2, 8])

✓ Found 2 detections

LAYER 0/1: module.backbone.ERBlock_2.0.rbr_dense.conv
  Prototypes: {0: 4, 1: 4}

  Detection 0/1: class=1, conf=0.840, n_prototypes=4
torch.Size([2, 8])


ERROR:LCRP.utils.crp:Prediction time: 0.06586503982543945, Backward time: 0.30121946334838867, Full attribution time: 0.30608296394348145



[DEBUG] PROTOTYPE SELECTION:
  closest_row in meta: 7060
  tile_ds_idx (exhaustive_dataset index): 6667
  proto_img_idx (PersonCarDataset index): 182
  box_idx_proto: 10
  meta entry: {'dataset_idx': 6667, 'img_idx': 182, 'tile_x0': 576, 'tile_y0': 0, 'box_idx': 10, 'cls': 1, 'conf': 0.7408130168914795, 'box': [264.8069152832031, 139.1837158203125, 282.8142395019531, 170.08935546875]}

torch.Size([24, 8])
[ATTR FIX] Using box_idx_proto=10 (IoU=1.000) for attribution
[DEBUG] original_shape_p (computed): (1080, 1920)
[DEBUG] tile shape: torch.Size([3, 640, 640])
[DEBUG CROP] predicted_box_p: [264.80692 139.18372 282.81424 170.08936]


ERROR:LCRP.utils.crp:Prediction time: 0.0841665267944336, Backward time: 0.1794588565826416, Full attribution time: 0.18418359756469727


torch.Size([2, 8])


...

torch.Size([10, 8])
taking prediction num.  8  (wanted  8 )
taking prediction num.  8  (wanted  8 )
taking prediction num.  8  (wanted  8 )


ERROR:LCRP.utils.crp:Prediction time: 0.03922891616821289, Backward time: 0.12658405303955078, Full attribution time: 0.13104701042175293


torch.Size([10, 8])
taking prediction num.  8  (wanted  8 )
➡️ prototype+concept figure: ../output_RAS/pcx/pcx_plots/module.backbone.ERBlock_3.0.rbr_dense.conv_class1_K4_proto_concepts.png
torch.Size([2, 8])
[DBG] prediction_num(draw)=1  mapped_to_attr_idx=1
torch.Size([2, 8])
[DBG] exporting HTML with test_predicted_box in input coords


/opt/conda/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



[OK] Exported interactive HTML to: ../output_RAS/pcx/export_html/gmm_export_layer_module_backbone_ERBlock_3_0_rbr_dense_conv_class_1_det01/index.html
Open it directly, or serve the folder with:  python -m http.server 8000  (then visit http://localhost:8000/gmm_export_layer_module_backbone_ERBlock_3_0_rbr_dense_conv_class_1_det01/index.html)
[OK] Saved 2D GMM plot: ../output_RAS/pcx/gmm_2d/module.backbone.ERBlock_3.0.rbr_dense.conv/gmm_2d_class1_K4.png
[LL] mixture p=0.253 | local p=0.434 | M2 p=0.857 | γ=1.00 | π=0.22 | coverage=21.8% | show_extra_diag=False
    ✅ Saved: ../output_RAS/pcx/pcx_plots/module_backbone_ERBlock_3_0_rbr_dense_conv/tile0600_box1_class1_layermodule_backbone_ERBlock_3_0_rbr_dense_conv.png

######################################################################
# ✓ DONE processing tile_idx=600
######################################################################


### 6C) PCX visualisation of a dataset: show all dataset examples (with all predictions) per prototype


In [10]:
# # import os
# from PIL import Image
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from crp.helper import get_layer_names
# from tqdm import tqdm

# # ============ CONFIGURATION ============
# n_prototypes_by_layer = {
#     "module.backbone.stem.rbr_dense.conv":   {0: 3, 1: 4},
#     "module.backbone.ERBlock_2.0.rbr_dense.conv":        {0: 3, 1: 3},
#     "module.backbone.ERBlock_3.0.rbr_dense.conv":        {0: 3, 1: 3},

# }

# layer_names = list(n_prototypes_by_layer.keys())

# use_half = False

# # ============ USE TRAINING DATASET ============
# # Change these to your training dataset variables
# train_dataset = dataset  # Preprocessed training dataset
# train_orig_dataset = orig_dataset  # Original training dataset

# # Get total number of training images
# num_train_images = len(train_orig_dataset)

# # Get stride from model (before moving)
# model = model.to(device)
# stride = int(model.stride.max()) if hasattr(model, 'stride') else 64
# img_size = check_img_size(640, stride=stride)

# print(f"Using image size: {img_size}, stride: {stride}")
# print(f"Total training images: {num_train_images}")
# print(f"\n✓ Processing {len(layer_names)} layers:")
# for ln in layer_names:
#     print(f"  - {ln}: {n_prototypes_by_layer[ln]}")

# # ============ STATISTICS TRACKING ============
# stats = {
#     "total_images": num_train_images,
#     "images_processed": 0,
#     "images_with_detections": 0,
#     "total_detections": 0,
#     "total_plots_saved": 0,
#     "errors": 0
# }

# # ============ ENSURE MODEL IS ON CORRECT DEVICE ============
# model = model.to(device)
# model.eval()

# # Enable gradients for model parameters (needed for CRP/PCX explanations)
# for param in model.parameters():
#     param.requires_grad_(True)

# # ============ MAIN LOOP: ALL TRAINING IMAGES ============
# for train_idx in tqdm(range(num_train_images), desc="Processing training images"):
#     print(f"\n{'#'*70}")
#     print(f"# TRAINING IMAGE {train_idx}/{num_train_images-1}")
#     print(f"{'#'*70}")

#     try:
#         # Get original image from train_orig_dataset
#         orig_img_raw, label = train_orig_dataset[train_idx]

#         # Convert to PIL if needed
#         if torch.is_tensor(orig_img_raw):
#             if orig_img_raw.dtype == torch.uint8:
#                 orig_img = Image.fromarray(orig_img_raw.permute(1, 2, 0).numpy())
#             else:
#                 orig_img = Image.fromarray((orig_img_raw.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
#         elif isinstance(orig_img_raw, np.ndarray):
#             orig_img = Image.fromarray(orig_img_raw)
#         else:
#             orig_img = orig_img_raw

#         orig_np = np.array(orig_img)
#         original_shape = orig_np.shape[:2]

#         print(f"Original image shape (HxW): {original_shape}")

#         # Apply letterbox preprocessing (auto=True for single image)
#         img_letterbox = letterbox(orig_np, new_shape=img_size, stride=stride, auto=True)[0]
#         letterbox_shape = img_letterbox.shape[:2]

#         print(f"Letterbox image shape (HxW): {letterbox_shape}")

#         # Convert to tensor
#         img_letterbox_transposed = img_letterbox.transpose((2, 0, 1))
#         img_tensor = torch.from_numpy(np.ascontiguousarray(img_letterbox_transposed))
#         img_tensor = img_tensor.float() if not use_half else img_tensor.half()
#         img_tensor /= 255.0
#         img_tensor = img_tensor.to(device)
#         img_tensor.requires_grad_(True)  # Enable gradients for CRP/PCX explanations

#         print(f"Preprocessed tensor shape: {img_tensor.shape}")

#         # Run prediction (WITH gradients for CRP/PCX explanations)
#         img_with_batch = img_tensor.unsqueeze(0)
#         img_with_batch.requires_grad_(True)  # Enable gradients for explanation methods
#         scores, boxes = model.predict_with_boxes(img_with_batch)

#         # Rescale boxes to original coordinates
#         boxes_original = rescale_boxes(
#             boxes[0].cpu().detach().numpy(),
#             letterbox_shape=letterbox_shape,
#             original_shape=original_shape
#         )

#         num_boxes = boxes_original.shape[0]
#         class_ids = scores[0].argmax(dim=1)
#         confidences = scores[0].max(dim=1).values

#         print(f"\n✓ Found {num_boxes} detections")
#         stats["images_processed"] += 1

#         if num_boxes == 0:
#             print("⚠️ No detections, skipping image")
#             continue

#         stats["images_with_detections"] += 1
#         stats["total_detections"] += num_boxes

#         # ============ LOOP THROUGH ALL BOUNDING BOXES ============
#         for prediction_num in range(num_boxes):
#             class_id = class_ids[prediction_num].item()
#             box_original = boxes_original[prediction_num]
#             conf = confidences[prediction_num].item()

#             print(f"\n  Processing detection {prediction_num}/{num_boxes-1}: "
#                   f"class={class_id}, conf={conf:.3f}")

#             # ============ LOOP THROUGH ALL LAYERS ============
#             for layer_idx, layer_name in enumerate(layer_names):
#                 print(f"\n    Layer {layer_idx}/{len(layer_names)-1}: {layer_name}")

#                 prototype_dict = n_prototypes_by_layer[layer_name]

#                 if class_id not in prototype_dict:
#                     print(f"      class={class_id} not in prototype_dict, skipping")
#                     continue

#                 print(f"      n_prototypes={prototype_dict[class_id]}")

#                 safe_layer = layer_name.replace(".", "_")
#                 out_dir = os.path.join(f"../output_{dataset_type}/pcx/pcx_plots", safe_layer)
#                 os.makedirs(out_dir, exist_ok=True)

#                 try:
#                     # Pass CPU tensor since plot_pcx_explanations uses CPU
#                     img_tensor_cpu = img_tensor.cpu()

#                     fig = plot_pcx_explanations(
#                         model_name=model_name,
#                         model=model,
#                         img=img_tensor_cpu,  # CPU tensor
#                         orig_img=orig_img,
#                         dataset=train_dataset,
#                         orig_dataset=train_orig_dataset,
#                         class_id=class_id,
#                         n_concepts=3,
#                         n_refimgs=12,
#                         num_prototypes=prototype_dict,
#                         prediction_num=prediction_num,
#                         layer_name=layer_name,
#                         ref_imgs_path=f"../output_{dataset_type}/ref_imgs/",
#                         output_dir_pcx=f"../output_{dataset_type}/pcx/yolo_person_car",
#                         output_dir_crp=output_dir_crp,
#                         letterbox_shape=letterbox_shape,
#                         original_shape=original_shape,
#                         rescale_boxes_fn=rescale_boxes,
#                         dataset_type=dataset_type
#                     )

#                     # Move model back to CUDA after plot_pcx_explanations
#                     model = model.to(device)

#                     if fig is not None:
#                         fname = f"train{train_idx:04d}_box{prediction_num}_class{class_id}_layer{safe_layer}.png"
#                         out_path = os.path.join(out_dir, fname)
#                         fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor='white')
#                         plt.close(fig)
#                         print(f"      ✅ Saved: {out_path}")
#                         stats["total_plots_saved"] += 1
#                     else:
#                         print(f"      ⚠️ No figure returned")

#                 except Exception as e:
#                     print(f"      ❌ Error: {e}")
#                     import traceback
#                     traceback.print_exc()
#                     stats["errors"] += 1
#                     # Ensure model is back on CUDA even after error
#                     model = model.to(device)
#                     continue

#     except Exception as e:
#         print(f"  ❌ Error processing image {train_idx}: {e}")
#         import traceback
#         traceback.print_exc()
#         stats["errors"] += 1
#         model = model.to(device)
#         continue

# # ============ FINAL STATISTICS ============
# print(f"\n{'#'*70}")
# print(f"# ✓ COMPLETED PROCESSING ALL TRAINING IMAGES")
# print(f"{'#'*70}")
# print(f"\n📊 Statistics:")
# print(f"  Total images in dataset: {stats['total_images']}")
# print(f"  Images processed: {stats['images_processed']}")
# print(f"  Images with detections: {stats['images_with_detections']}")
# print(f"  Total detections: {stats['total_detections']}")
# print(f"  Total plots saved: {stats['total_plots_saved']}")
# print(f"  Errors encountered: {stats['errors']}")